# Composable Overlay Introduction
----

<div class="alert alert-box alert-info">
Please use Jupyter labs http://&lt;board_ip_address&gt;/lab for this notebook.
</div>

This notebook provides an introductory overview of the Composable Overlay and how to use it from a Jupyter Notebook

## Aims
* Describe the Composable Pipeline architecture
* Explore the `Composable` class

## Table of Contents
* [Introduction](#intro)
* [The Composable Class](#composable_class)
* [Download the Composable Overlay](#download)
* [Explore Composable Dictionary](#c_dict)
* [Explore DFX Dictionary](#dfx_dict)
* [Documentation](#docs)
* [Conclusion](#conclusion)

## Revision History

* v1.0 | 30 March 2021 | First notebook revision.
* v1.1 | 11 August 2021 | Update notebook to composable overlay API 1.0.0

----

## Introduction <a class="anchor" id="intro"></a>

A Composable Overlay is an overlay with a novel and clever architecture that allow us to adapt how the data flows between the IP cores connected to the switch.

The key characteristic of a composable overlay is an AXI4-Stream Switch at its core, which plays the same role as an old [telephone switchboard](https://en.wikipedia.org/wiki/Telephone_switchboard). The AXI4-Stream Switch provides the runtime routing of the data flow from one IP to another in a one to one manner. 

On top of that, the composable overlay includes DFX (partial reconfigurable) regions that can bring new functionality to the design.

The composable overlay architecture of a composable video overlay for the PYNQ-Z2 is shown in the image below

![](../img/cv-4pr.png)

## The Composable Class <a class="anchor" id="composable_class"></a>

The `Composable` class inherits from the pynq [`DefaultHierarchy` class](https://pynq.readthedocs.io/en/latest/pynq_package/pynq.overlay.html?#pynq.overlay.DefaultHierarchy) and enhances it by adding new methods and attributes to compose pipelines at run time.
The `Composable` class abstracts the Composable Overlay logic in a pythonic way, pynq assigns this class (driver) to any hierarchy that contains and [AXI4-Stream Switch](https://www.xilinx.com/products/intellectual-property/axi4-stream_interconnect.html#documentation) (`axis_switch`) and [AXI GPIO](https://www.xilinx.com/products/intellectual-property/axi_gpio.html#documentation) for soft reset (`pipeline_control`). 

## Download the Composable Overlay <a class="anchor" id="download"></a>

It is time to download the composable overlay using the `pynq.Overlay` class and grab a handler to the `.composable` hierarchy.

pynq will automatically assign the `Composable` class to the `.composable` hierarchy given that its meets requirements.

The `Overlay` class will:

1. Download the bitstream onto the FPGA


In [1]:
from pynq import Overlay
import pynq_composable

ol = Overlay("cv_dfx_3_pr.bit")

In [13]:
ol.hierarchy_dict

{'composable': {'ip': {'axis_switch': {'type': 'xilinx.com:ip:axis_switch:1.1',
    'mem_id': 'S_AXI_CTRL',
    'memtype': 'REGISTER',
    'gpio': {},
    'interrupts': {},
    'parameters': {'C_FAMILY': 'zynquplus',
     'C_NUM_SI_SLOTS': '16',
     'C_LOG_SI_SLOTS': '4',
     'C_NUM_MI_SLOTS': '15',
     'C_AXIS_TDATA_WIDTH': '24',
     'C_AXIS_TID_WIDTH': '1',
     'C_AXIS_TDEST_WIDTH': '1',
     'C_AXIS_TUSER_WIDTH': '1',
     'C_AXIS_SIGNAL_SET': '0b00000000000000000000000010010011',
     'C_ARB_ON_MAX_XFERS': '1',
     'C_ARB_ON_NUM_CYCLES': '0',
     'C_ARB_ON_TLAST': '0',
     'C_INCLUDE_ARBITER': '1',
     'C_ARB_ALGORITHM': '0',
     'C_OUTPUT_REG': '0',
     'C_DECODER_REG': '1',
     'C_M_AXIS_CONNECTIVITY_ARRAY': '0b111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111',
     'C_M_AXIS_BAS

We can verify that pynq has assigned the `Composable` class by checking pynq `.hierarchy_dict`

In [2]:
ol.hierarchy_dict['composable']

{'ip': {'axis_switch': {'type': 'xilinx.com:ip:axis_switch:1.1',
   'mem_id': 'S_AXI_CTRL',
   'memtype': 'REGISTER',
   'gpio': {},
   'interrupts': {},
   'parameters': {'C_FAMILY': 'zynquplus',
    'C_NUM_SI_SLOTS': '16',
    'C_LOG_SI_SLOTS': '4',
    'C_NUM_MI_SLOTS': '15',
    'C_AXIS_TDATA_WIDTH': '24',
    'C_AXIS_TID_WIDTH': '1',
    'C_AXIS_TDEST_WIDTH': '1',
    'C_AXIS_TUSER_WIDTH': '1',
    'C_AXIS_SIGNAL_SET': '0b00000000000000000000000010010011',
    'C_ARB_ON_MAX_XFERS': '1',
    'C_ARB_ON_NUM_CYCLES': '0',
    'C_ARB_ON_TLAST': '0',
    'C_INCLUDE_ARBITER': '1',
    'C_ARB_ALGORITHM': '0',
    'C_OUTPUT_REG': '0',
    'C_DECODER_REG': '1',
    'C_M_AXIS_CONNECTIVITY_ARRAY': '0b111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111',
    'C_M_AXIS_BASETDEST_ARRAY': '0b010101010101010',
 

In [3]:
ol?

Grab a handle to the composable hierarchy this will:
1. Perform hardware discovery of the IP cores connected to the AXI4-Stream Switch (either by checking cached file or run the discovery on-the-fly)
1. Discovery the DFX regions available in the ComposableOverlay
1. Configure the AXI4-Stream Switch with the default values

In [4]:
cpipe = ol.composable

## Explore Composable Dictionary <a class="anchor" id="c_dict"></a>

The `.c_dict` attribute displays all of the IP cores available to compose our pipeline. 

These IP cores can be on the static region or in the DFX regions. IP cores on the static region are `[loaded]` by default whereas the IP cores in the DFX regions are `[unloaded]` by default. The `[default]` attribute specify paths that are producer and consumers of data in a given pipeline. These defaults paths, and the path name, can be specified with a [JSON](../overlay/cv_dfx_4_pr_paths.json) file.

If you expand an entry on this dictionary, you will see
- ci: (consumer interface) the output stream of this IP is connected to this consumer interface on the AXI4-Stream Switch
- pi: (producer interface) the input stream of this IP is connected from this producer interface on the AXI4-Stream Switch
- dfx: whether or not the IP is within a DFX region
- loaded: whether or not the IP is loaded
- default: (optional) whether the IP is the producer or consumer of a given pipeline

In [5]:
cpipe.c_dict

{'filter2d_accel': {'si': [2],
  'dfx': False,
  'loaded': True,
  'modtype': 'filter2d_accel',
  'mi': [2]},
 'lut_accel': {'si': [3],
  'dfx': False,
  'loaded': True,
  'modtype': 'lut_accel',
  'mi': [3]},
 'rgb2gray_accel': {'si': [4],
  'dfx': False,
  'loaded': True,
  'modtype': 'rgb2gray_accel',
  'mi': [4]},
 'gray2rgb_accel': {'si': [5],
  'dfx': False,
  'loaded': True,
  'modtype': 'gray2rgb_accel',
  'mi': [5]},
 'rgb2hsv_accel': {'si': [6],
  'dfx': False,
  'loaded': True,
  'modtype': 'rgb2hsv_accel',
  'mi': [6]},
 'colorthresholding_accel': {'si': [7],
  'dfx': False,
  'loaded': True,
  'modtype': 'colorthresholding_accel',
  'mi': [7]},
 'duplicate_accel': {'si': [8, 9],
  'dfx': False,
  'loaded': True,
  'modtype': 'axis_broadcaster',
  'mi': [8]},
 'pr_0/dilate_accel': {'mi': [9],
  'si': [10],
  'modtype': 'dilate_accel',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_0/erode_accel': {'mi': [10],
 

You can also filter by *loaded*, *unloaded* and *default* IP cores using the `.loaded` and `.unloaded` attributes of the `c_dict`

In [6]:
cpipe.c_dict.loaded

{'filter2d_accel': {'si': [2],
  'dfx': False,
  'loaded': True,
  'modtype': 'filter2d_accel',
  'mi': [2]},
 'lut_accel': {'si': [3],
  'dfx': False,
  'loaded': True,
  'modtype': 'lut_accel',
  'mi': [3]},
 'rgb2gray_accel': {'si': [4],
  'dfx': False,
  'loaded': True,
  'modtype': 'rgb2gray_accel',
  'mi': [4]},
 'gray2rgb_accel': {'si': [5],
  'dfx': False,
  'loaded': True,
  'modtype': 'gray2rgb_accel',
  'mi': [5]},
 'rgb2hsv_accel': {'si': [6],
  'dfx': False,
  'loaded': True,
  'modtype': 'rgb2hsv_accel',
  'mi': [6]},
 'colorthresholding_accel': {'si': [7],
  'dfx': False,
  'loaded': True,
  'modtype': 'colorthresholding_accel',
  'mi': [7]},
 'duplicate_accel': {'si': [8, 9],
  'dfx': False,
  'loaded': True,
  'modtype': 'axis_broadcaster',
  'mi': [8]},
 'ps_video_in': {'si': [0],
  'dfx': False,
  'loaded': True,
  'modtype': 'pixel_unpack_2',
  'default': True,
  'fullpath': 'video/pixel_unpack'},
 'ps_video_out': {'mi': [0],
  'dfx': False,
  'loaded': True,
  'mod

In [7]:
cpipe.c_dict.unloaded

{'pr_0/dilate_accel': {'mi': [9],
  'si': [10],
  'modtype': 'dilate_accel',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_0/erode_accel': {'mi': [10],
  'si': [11],
  'modtype': 'erode_accel',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_0/filter2d_accel': {'mi': [9],
  'si': [10],
  'modtype': 'filter2d_accel',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_filter2d_fifo_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_0/axis_data_fifo_0': {'mi': [10],
  'si': [11],
  'modtype': 'axis_data_fifo',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_fast_fifo_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_0/fast_accel': {'mi': [9],
  'si': [10],
  'modtype': 'fast_accel',
  'bitstream': 'cv_dfx_3_pr_composable_pr_0_fast_fifo_partial.bit',
  'dfx': True,
  'loaded': False},
 'pr_1/cornerHarris_accel': {'mi': [11],
  'si': [12],
  'modtype': 'cornerHarris_ac

In [8]:
cpipe.c_dict.default

{'ps_video_in': {'si': [0],
  'dfx': False,
  'loaded': True,
  'modtype': 'pixel_unpack_2',
  'default': True,
  'fullpath': 'video/pixel_unpack'},
 'ps_video_out': {'mi': [0],
  'dfx': False,
  'loaded': True,
  'modtype': 'pixel_pack_2',
  'default': True,
  'fullpath': 'video/pixel_pack'},
 'mipi_in': {'si': [1],
  'dfx': False,
  'loaded': True,
  'modtype': 'v_proc_ss',
  'default': True,
  'fullpath': 'mipi/v_proc_sys'},
 'mipi_out': {'mi': [1],
  'dfx': False,
  'loaded': True,
  'modtype': 'pixel_pack_2',
  'default': True,
  'fullpath': 'mipi/pixel_pack'}}

## Explore DFX Dictionary <a class="anchor" id="dfx_dict"></a>

The `.dfx_dict` attribute displays all of the DFX regions available in the composable overlay

If you expand an entry on this dictionary, you will see
- decoupler: name of the decoupler that handles the DFX regions
- gpio: PS GPIO pins that enable decouple and status of said decoupler
- rm: (reconfigurable module) dictionary of partial bitstreams and IP cores contained in them

In [9]:
cpipe.dfx_dict

{'pr_0': {'decoupler': '/composable/dfx_decouplers/dfx_decoupler_pr_0',
  'decouple': 0,
  'status': 3,
  'rm': {'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit': {'pr_0/dilate_accel': {'interface': ['/pr_0/stream_in0',
      '/pr_0/stream_out0'],
     'modtype': 'dilate_accel',
     'bitstream': 'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit'},
    'pr_0/erode_accel': {'interface': ['/pr_0/stream_in1',
      '/pr_0/stream_out1'],
     'modtype': 'erode_accel',
     'bitstream': 'cv_dfx_3_pr_composable_pr_0_dilate_erode_partial.bit'}},
   'cv_dfx_3_pr_composable_pr_0_filter2d_fifo_partial.bit': {'pr_0/filter2d_accel': {'interface': ['/pr_0/stream_in0',
      '/pr_0/stream_out0'],
     'modtype': 'filter2d_accel',
     'bitstream': 'cv_dfx_3_pr_composable_pr_0_filter2d_fifo_partial.bit'},
    'pr_0/axis_data_fifo_0': {'interface': ['/pr_0/stream_in1',
      '/pr_0/stream_out1'],
     'modtype': 'axis_data_fifo',
     'bitstream': 'cv_dfx_3_pr_composable_pr_0_filter2d_fifo_p

## Documentation <a class="anchor" id="docs"></a>

You can get more detailed documentation on any of the classes described above by using the built-in `help()` function. Or using the question mark (`?`)

In [14]:
from pynq_composable import Composable

In [15]:
Composable.compose?

In [16]:
help(Composable)

Help on class Composable in module pynq_composable.composable:

class Composable(pynq.overlay.DefaultHierarchy)
 |  Composable(description)
 |  
 |  This class keeps track of a composable overlay
 |  
 |  The Composable class holds the state of the logic available for run-time
 |  composition through and AXI4-Stream switch
 |  
 |  Our definition of composable overlay is: "post-bitstream configurable
 |  dataflow pipeline". Hence, this class must expose configurability through
 |  content discovery, dataflow configuration and runtime protection.
 |  
 |  This class stores two dictionaries: c_dict and dfx_dict
 |  
 |  Each entry of the Composable dictionary (c_dict) is a mapping:
 |  'name' -> {si, mi, modtype, dfx, loaded, bitstream}, where
 |  name (str) is the key of the entry.
 |  si (list) list of physical port in the switch the IP is connected to
 |  mi (list) list of physical port in the switch the IP is connected from
 |  modtype(str) IP type
 |  dfx (bool) IP is located in a D

## Conclusion <a class="anchor" id="conclusion"></a>

This notebook has introduced the `Composable` class as well as `c_dict` and `dfx_dict`. We will use these dictionaries in the following notebooks to create custom pipelines

| | [First Custom Pipeline ➡️](02_first_custom_pipeline.ipynb)

Copyright &copy; 2021 Xilinx, Inc

SPDX-License-Identifier: BSD-3-Clause

----